# CardioIA — Fase 2 · Parte 2
## Classificador de risco: TF-IDF + Machine Learning

Aqui o sistema deixa de perguntar *qual doença* e passa a perguntar **com que urgência
atender**. É a lógica de uma triagem: a frase do paciente entra, e sai "alto risco" ou
"baixo risco" — quem espera e quem passa na frente.

O pipeline é o pedido no enunciado: **TF-IDF** para transformar frase em vetor numérico
e um **classificador simples do Scikit-learn** treinado sobre esses vetores.

A avaliação é a parte que leva mais espaço neste notebook, de propósito. Com uma base
pequena e escrita pela própria equipe, é trivial produzir uma acurácia bonita e
enganosa; o trabalho de verdade é descobrir onde o modelo falha.

A lógica vive em `src/classificador.py` e roda também por linha de comando
(`python -m src.classificador`).

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.classificador import (
    aplicar_limiar,
    avaliar,
    carregar_base,
    construir_pipeline,
    curva_limiar,
    prever,
    termos_influentes,
    testar_desafio,
)
from src.extracao import normalizar
from src.vieses_fase1 import carregar_pacientes, doentes_assintomaticos, prevalencia_por

pd.set_option("display.max_colwidth", 80)

---
## 1. A base rotulada

`data/frases_risco.csv`, no formato `frase,situacao` pedido no enunciado: **70 frases,
35 de cada classe**. O balanceamento é intencional — numa base desbalanceada, acurácia
alta pode significar apenas que o modelo aprendeu a chutar sempre a classe maior.

Duas decisões na hora de escrever as frases:

- **"Alto risco" não é sinônimo de "fala em peito".** Metade das frases de baixo risco
  também menciona peito, dor ou falta de ar ("senti uma fisgada rápida no peito ao me
  virar na cama"). Sem essa sobreposição, o problema viraria a busca por uma palavra e
  a avaliação não mediria nada.
- **Gravidade real, não vocabulário dramático.** O rótulo segue o critério clínico
  (duração, irradiação, sintomas associados, relação com esforço), não a intensidade do
  adjetivo.

In [2]:
base = carregar_base()
print(base.situacao.value_counts().to_string(), "\n")
display(base.groupby("situacao").head(4)[["frase", "situacao"]])

situacao
alto risco     35
baixo risco    35 



,frase,situacao
0,sinto dor no peito e falta de ar,alto risco
1,tenho uma dor forte no peito que irradia para o braço esquerdo,alto risco
2,acordei com aperto no peito e suor frio,alto risco
3,estou com dor no peito há quarenta minutos e não passa,alto risco
35,tive um leve incômodo nas costas,baixo risco
36,senti uma fisgada rápida no peito ao me virar na cama que durou dois segundos,baixo risco
37,estou com dor muscular no ombro depois de carregar caixas,baixo risco
38,minha perna doeu um pouco depois da caminhada de ontem,baixo risco


---
## 2. TF-IDF: da frase para o vetor

O algoritmo de ML não lê texto, lê números. O TF-IDF converte cada frase num vetor onde
cada posição é um termo, com peso maior para termos **frequentes naquela frase** e
**raros no resto da base** — é o que faz "irradia" pesar mais que "que".

Usamos `ngram_range=(1, 2)`, ou seja, palavras isoladas **e** pares. Isso importa
clinicamente: "falta" e "ar" separados dizem pouco; o bigrama "falta de ar" é o sinal.

In [3]:
pipe = construir_pipeline("logistica").fit(base.frase_norm, base.situacao)
tfidf = pipe.named_steps["tfidf"]
X = tfidf.transform(base.frase_norm)

print(f"matriz: {X.shape[0]} frases x {X.shape[1]} termos "
      f"({X.nnz / (X.shape[0] * X.shape[1]):.1%} preenchida — vetores esparsos)\n")

frase = base.frase_norm.iloc[0]
vetor = pd.Series(X[0].toarray()[0], index=tfidf.get_feature_names_out())
print("frase:", frase)
display(vetor[vetor > 0].sort_values(ascending=False).head(8).to_frame("peso TF-IDF"))

matriz: 70 frases x 742 termos (2.4% preenchida — vetores esparsos)

frase: sinto dor no peito e falta de ar


,peso TF-IDF
peito falta,0.428493
sinto dor,0.342571
de ar,0.298498
falta de,0.298498
ar,0.287453
falta,0.287453
sinto,0.287453
dor no,0.260477


---
## 3. Treino, teste e três modelos

70% treino / 30% teste, **estratificado** (mantém 50/50 nas duas partes) e com semente
fixa, para que qualquer pessoa reproduza os mesmos números.

Comparamos três modelos, e o primeiro é o mais importante: o **classificador burro**
(`DummyClassifier`), que responde sempre a classe mais frequente. Ele é a régua — um
modelo que não supera o burro não aprendeu nada, por mais bonita que seja sua acurácia.

Além da acurácia no teste, rodamos **validação cruzada 5-fold**: com 21 frases de teste,
um único split é ruído. O desvio-padrão entre os folds diz o quanto o número é instável.

In [4]:
comparacao = []
for modelo in ("burro", "arvore", "logistica"):
    r = avaliar(modelo, base)
    comparacao.append({
        "modelo": {"burro": "DummyClassifier (linha de base)",
                   "arvore": "DecisionTree",
                   "logistica": "LogisticRegression"}[modelo],
        "acurácia (teste)": round(r["acuracia_teste"], 3),
        "acurácia (CV 5-fold)": round(r["cv_media"], 3),
        "desvio entre folds": round(r["cv_desvio"], 3),
    })
display(pd.DataFrame(comparacao))

,modelo,acurácia (teste),acurácia (CV 5-fold),desvio entre folds
0,DummyClassifier (linha de base),0.476,0.500,0.000
1,DecisionTree,0.571,0.657,0.146
2,LogisticRegression,0.762,0.829,0.107


A regressão logística vence a linha de base com folga e supera a árvore de decisão — o
esperado com texto vetorizado: a árvore corta um termo por vez e sofre com espaço
esparso de centenas de dimensões, enquanto a logística soma a evidência de todos os
termos ao mesmo tempo.

O desvio entre folds (**±0,11**) é grande. Com 70 frases, uma diferença de 5 pontos de
acurácia entre modelos está dentro do ruído — não se deve declarar vencedor por margem
pequena numa base deste tamanho.

---
## 4. Onde o modelo erra — e por que o tipo de erro importa mais que a acurácia

Numa triagem, os dois erros **não custam a mesma coisa**:

- **Falso positivo** (baixo risco classificado como alto): alguém sem urgência entra na
  fila prioritária. Custo: tempo.
- **Falso negativo** (alto risco classificado como baixo): alguém infartando é mandado
  para a fila comum. Custo: a vida do paciente.

Por isso a métrica que governa este problema é o **recall de "alto risco"** — a fração
dos casos graves que o sistema conseguiu capturar —, não a acurácia global.

In [5]:
r = avaliar("logistica", base)
print(r["relatorio"])
display(r["matriz"])

              precision    recall  f1-score   support

  alto risco       0.80      0.73      0.76        11
 baixo risco       0.73      0.80      0.76        10

    accuracy                           0.76        21
   macro avg       0.76      0.76      0.76        21
weighted avg       0.77      0.76      0.76        21



,previsto: alto risco,previsto: baixo risco
real: alto risco,8,3
real: baixo risco,2,8


In [6]:
# Matriz de confusão — magnitude em uma única escala de cor (claro -> escuro),
# valores anotados em cada célula: a cor ordena, o número informa.
matriz = r["matriz"]
rotulos = ["alto risco", "baixo risco"]

fig, ax = plt.subplots(figsize=(4.6, 4.0))
img = ax.imshow(matriz.values, cmap="Blues", vmin=0, vmax=matriz.values.max())

for i in range(matriz.shape[0]):
    for j in range(matriz.shape[1]):
        v = matriz.values[i, j]
        ax.text(j, i, v, ha="center", va="center", fontsize=15,
                color="white" if v > matriz.values.max() * 0.6 else "#1f2933")

ax.set_xticks(range(2), [f"previsto\n{l}" for l in rotulos])
ax.set_yticks(range(2), [f"real\n{l}" for l in rotulos])
ax.set_title("Matriz de confusão — 21 frases de teste", pad=12, fontsize=11)
ax.tick_params(length=0, labelsize=9)
for lado in ax.spines.values():
    lado.set_visible(False)
fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04).outline.set_visible(False)
fig.tight_layout()
Path("../docs").mkdir(exist_ok=True)
fig.savefig("../docs/matriz_confusao.png", dpi=160, bbox_inches="tight")
plt.show()

<Figure size 460x400 with 2 Axes>

Os erros do modelo **não estão concentrados no lado seguro**: parte dos casos de alto
risco foi classificada como baixo risco. Num sistema real de triagem isso é
inaceitável sem um ajuste de limiar, que é o que fazemos a seguir.

### 4.1 Ajuste do limiar de decisão

Por padrão o modelo diz "alto risco" quando a probabilidade passa de 0,5. Esse corte
trata os dois erros como se custassem o mesmo, e na triagem eles não custam. Baixar o
corte faz o sistema alertar mais cedo: captura mais casos graves e, em troca, manda mais
gente sem urgência para a fila prioritária.

Para escolher o corte **sem olhar o teste**, usamos as probabilidades da validação
cruzada: cada uma das 70 frases é pontuada por um modelo que não a viu no treino.
Escolher o limiar olhando as 21 frases de teste seria ajustar o corte ao acaso daquele
sorteio.

In [7]:
display(curva_limiar(base))

,limiar,recall alto risco,graves perdidos,falsos positivos,acurácia
0,0.50,0.829,6,6,0.829
1,0.45,0.943,2,14,0.771
2,0.40,0.971,1,20,0.700
3,0.35,1.000,0,30,0.571


Com o corte em **0,45**, o sistema deixa de perder 6 dos 35 casos graves e passa a
perder 2: o recall de alto risco vai de 0,83 para 0,94. O preço é de 6 para 14 falsos
positivos, e a acurácia cai de 0,83 para 0,77. **Numa triagem, é a troca certa:** o
falso positivo custa tempo de fila, o falso negativo custa o paciente. Escolher o modelo
pela acurácia teria escolhido o corte errado.

**Uma ressalva sobre esses números:** eles vêm das mesmas probabilidades usadas para
escolher o corte, então são otimistas. Abaixo, o corte de 0,45 aplicado às 21 frases do
teste, com o modelo treinado só nas outras 49. Também não é uma medida independente (as
21 frases participaram da validação cruzada que escolheu o corte), e com 70 frases não dá
para separar um terceiro conjunto sem esvaziar os outros. A medida realmente
independente é o conjunto-desafio da seção 6.

Baixar mais (0,35) zera os casos perdidos mas dispara alerta para quase todo mundo (30
falsos positivos em 35), e uma triagem que prioriza todos não prioriza ninguém.

In [8]:
import numpy as np

frases_teste = base.loc[r["y_teste"].index, "frase"].tolist()
y_teste = r["y_teste"].to_numpy()
for corte in (0.5, 0.45):
    previsto = np.array(aplicar_limiar(r["pipeline"], frases_teste, corte))
    grave, alerta = y_teste == "alto risco", previsto == "alto risco"
    print(f"corte {corte}: {(alerta & grave).sum()}/{grave.sum()} graves capturados · "
          f"{(alerta & ~grave).sum()}/{(~grave).sum()} falsos positivos · "
          f"acurácia {(previsto == y_teste).mean():.3f}")

corte 0.5: 8/11 graves capturados · 2/10 falsos positivos · acurácia 0.762
corte 0.45: 11/11 graves capturados · 3/10 falsos positivos · acurácia 0.857


---
## 5. O que o modelo aprendeu (e por que isso é preocupante)

A regressão logística é interpretável: cada termo tem um peso que empurra a decisão para
um lado. Isso permite auditar o modelo — algo que uma rede neural não entregaria de graça.

In [9]:
display(termos_influentes(pipe, n=10))

,puxa p/ 'alto risco',puxa p/ 'baixo risco'
0,no peito,um
1,peito,depois
2,dor no,leve
3,sinto,senti
4,falta,depois de
5,ar,pouco
6,para,um pouco
7,dor,do
8,falta de,ao
9,de ar,estou com


Os termos que puxam para **alto risco** são clinicamente plausíveis: "no peito", "falta
de ar", "dor". Mas os que puxam para **baixo risco** revelam o problema: `leve`,
`pouco`, `depois`, `um pouco`. O modelo não aprendeu gravidade clínica — aprendeu que
**quem escreve "leve" está bem**. É uma correlação real na nossa base e uma regra
perigosa no mundo: "tive um leve incômodo no peito o dia inteiro" descreve um quadro
grave com vocabulário ameno.

---
## 6. O conjunto-desafio: testando o modelo onde ele deve falhar

Acurácia num teste sorteado da mesma base mede consistência interna. Para medir alguma
coisa parecida com generalização, escrevemos `data/frases_desafio.csv`: 8 frases que o
modelo **nunca viu** e que atacam de propósito uma fraqueza conhecida cada — negação,
apresentação atípica, vocabulário clínico ausente do treino, causa benigna com palavras
de risco.

In [10]:
desafio = testar_desafio(pipe)
display(desafio[["frase", "esperado", "previsto", "p(alto risco)", "acertou", "o_que_testa"]])
print(f"Acertos no desafio: {desafio.acertou.sum()}/{len(desafio)} "
      f"— contra {r['acuracia_teste']:.0%} no teste sorteado da própria base.")

,frase,esperado,previsto,p(alto risco),acertou,o_que_testa
0,"não sinto dor no peito nem falta de ar, só vim renovar a receita",baixo risco,alto risco,0.750,False,negação: o modelo vê as palavras de risco e ignora o 'não'
1,não tenho mais aquela dor no peito que eu sentia semana passada,baixo risco,alto risco,0.631,False,negação no passado
2,"estou com um cansaço estranho, enjoo e uma dor entre as costas há dois dias",alto risco,baixo risco,0.480,False,apresentação atípica (mais comum em mulheres): infarto sem a palavra 'peito'
3,sinto uma opressão retroesternal aos esforços,alto risco,alto risco,0.555,True,vocabulário clínico ausente do treino
4,meu filho me deu um soco no peito brincando e ficou doendo no lugar,baixo risco,alto risco,0.553,False,causa traumática benigna com vocabulário de risco
5,tive um leve incômodo no peito que durou o dia inteiro e piora quando ando,alto risco,baixo risco,0.431,False,vocabulário de baixo risco descrevendo quadro grave
6,sinto o coração acelerado há três horas sem parar,alto risco,alto risco,0.623,True,paráfrase de um caso de treino
7,"minha vizinha teve um infarto e eu fiquei preocupado, mas estou bem",baixo risco,baixo risco,0.423,True,menção a doença grave sem sintoma próprio


Acertos no desafio: 3/8 — contra 76% no teste sorteado da própria base.


**O desabamento de ~76% para 38% é o resultado mais importante deste notebook.** Ele
mostra que a acurácia do teste media a homogeneidade da base, não a competência do
modelo. Os três padrões de erro:

**1. Negação — a falha mais grave.** *"Não sinto dor no peito nem falta de ar"* é
classificada como **alto risco** com 75% de confiança. Para o TF-IDF, "não" é só mais um
token; a frase contém "dor no peito" e "falta de ar", e é isso que decide. O extrator de
regras da **Parte 1 acerta este caso** — ele checa negação explicitamente. Nenhuma das
duas abordagens domina a outra.

**2. Apresentação atípica.** *"Cansaço estranho, enjoo e dor entre as costas"* é uma
descrição clássica de infarto **sem dor torácica típica** — mais frequente em mulheres,
e a razão documentada de subdiagnóstico feminino. O modelo classifica como baixo risco.
Este não é um erro do algoritmo: é a nossa base que quase não tem esse padrão. O viés
entrou pelos dados.

**Com o limiar de 0,45**, o desafio sobe de 3/8 para 4/8: a apresentação atípica
(probabilidade 0,48) passa a ser alertada. A negação continua errada, porque ali a
probabilidade de alto risco é *alta* (0,75); nenhum ajuste de corte resolve um erro que
está na representação do texto.

**3. Vocabulário fora do treino.** "Opressão retroesternal" passou; mas qualquer termo
que não esteja nas 70 frases é invisível — o TF-IDF não tem noção de sinônimo.

In [11]:
desafio_045 = testar_desafio(pipe, limiar=0.45)
print(f"Desafio com limiar 0,45: {desafio_045.acertou.sum()}/{len(desafio_045)}")

Desafio com limiar 0,45: 4/8


---
## 7. O que acontece quando se corrige a base em vez do modelo

Se o problema é cobertura da base, a correção é dado, não hiperparâmetro. Adicionamos
8 frases (`data/frases_ampliacao.csv`) cobrindo negação e apresentação atípica e
reavaliamos tudo. O conjunto-desafio continua fora do treino. Uma ressalva honesta: as
frases da ampliação foram escritas depois de vermos os erros do desafio, e uma delas ("não
estou com dor no peito nem falta de ar, só preciso de um atestado") é quase uma paráfrase
da primeira frase do desafio.

In [12]:
ampliacao = pd.read_csv("../data/frases_ampliacao.csv")
ampliada = pd.concat([base[["frase", "situacao"]], ampliacao], ignore_index=True)
ampliada["frase_norm"] = ampliada.frase.map(normalizar)

r_amp = avaliar("logistica", ampliada)
pipe_amp = construir_pipeline("logistica").fit(ampliada.frase_norm, ampliada.situacao)
desafio_amp = testar_desafio(pipe_amp)

display(pd.DataFrame([
    {"base": "original (70 frases)", "acurácia (teste)": round(r["acuracia_teste"], 3),
     "acurácia (CV)": round(r["cv_media"], 3), "desafio": f"{desafio.acertou.sum()}/8"},
    {"base": "ampliada (78 frases)", "acurácia (teste)": round(r_amp["acuracia_teste"], 3),
     "acurácia (CV)": round(r_amp["cv_media"], 3), "desafio": f"{desafio_amp.acertou.sum()}/8"},
]))

# Frase a frase: o que a ampliação mudou de fato.
display(pd.DataFrame({
    "frase": desafio.frase.str[:60],
    "esperado": desafio.esperado,
    "p(alto) original": desafio["p(alto risco)"],
    "p(alto) ampliada": desafio_amp["p(alto risco)"],
    "acertou antes": desafio.acertou,
    "acertou depois": desafio_amp.acertou,
}))

,base,acurácia (teste),acurácia (CV),desafio
0,original (70 frases),0.762,0.829,3/8
1,ampliada (78 frases),0.625,0.745,4/8


,frase,esperado,p(alto) original,p(alto) ampliada,acertou antes,acertou depois
0,"não sinto dor no peito nem falta de ar, só vim renovar a rec",baixo risco,0.750,0.575,False,False
1,não tenho mais aquela dor no peito que eu sentia semana pass,baixo risco,0.631,0.598,False,False
2,"estou com um cansaço estranho, enjoo e uma dor entre as cost",alto risco,0.480,0.542,False,True
3,sinto uma opressão retroesternal aos esforços,alto risco,0.555,0.581,True,True
4,meu filho me deu um soco no peito brincando e ficou doendo n,baixo risco,0.553,0.537,False,False
5,tive um leve incômodo no peito que durou o dia inteiro e pio,alto risco,0.431,0.439,False,False
6,sinto o coração acelerado há três horas sem parar,alto risco,0.623,0.614,True,True
7,"minha vizinha teve um infarto e eu fiquei preocupado, mas es",baixo risco,0.423,0.437,True,True


A acurácia caiu (0,83 → 0,75 na validação cruzada) porque as 8 frases novas são casos
difíceis e tornam a base menos homogênea. Os números de teste, 0,762 → 0,625, comparam
sorteios diferentes e por isso pesam menos que a validação cruzada.

O ganho no desafio, olhado frase a frase, é **pequeno e diz muito**:

- **Só uma frase virou**: a apresentação atípica (cansaço, enjoo, dor nas costas) passou
  de 0,48 para 0,54 e cruzou o corte. É um acerto na margem, não uma mudança de
  comportamento. Com 8 frases, 3/8 → 4/8 é um sinal, não uma prova.
- **A negação não foi aprendida.** Mesmo com 4 frases novas de negação no treino, "não
  sinto dor no peito nem falta de ar" continua alto risco (0,75 → 0,58). O TF-IDF trata a
  frase como um saco de palavras: "não" é um termo a mais, sem relação com o termo que
  vem depois dele. Exemplos ajudam pouco quando o erro é da representação. É exatamente o
  que a regra da Parte 1 resolve (ela olha a palavra antes da expressão), e o argumento
  mais forte para combinar as duas abordagens.

**O que fica:** corrigir a base derrubou a métrica interna e moveu o desafio só na margem.
A acurácia sozinha teria descartado a mudança; o desafio sozinho, com 8 frases, também
não prova que ela funciona. As duas medidas juntas dizem o que se pode afirmar.

---
## 8. Viés e governança: o que este sistema não deve fazer

Quatro vieses que conseguimos nomear na nossa própria base:

| Viés | De onde vem | Efeito |
|---|---|---|
| **Apresentação atípica sub-representada** | Nossas frases seguem o padrão "dor no peito com irradiação" — o quadro clássico masculino | Mulheres com infarto atípico são classificadas como baixo risco (demonstrado na seção 6) |
| **Vocabulário como proxy de gravidade** | "leve" e "pouco" só aparecem em frases de baixo risco | Quem minimiza a própria queixa é despriorizado — e minimizar queixa é comportamento associado a idosos e a pacientes com menos acesso a serviço de saúde |
| **Base autoral** | As 70 frases foram escritas pela mesma equipe que rotulou | O modelo aprende o nosso jeito de escrever, não o jeito do paciente |
| **Ausência de negação no treino** | Nenhuma frase original diz "não sinto" | Consulta de rotina vira urgência |


### 8.1 O dataset da Fase 1 contra o nosso classificador

Na Fase 1 a equipe levantou o subconjunto Cleveland do **UCI Heart Disease**
([repositório da Fase 1](https://github.com/davisjr2000/cardio-ia)): 303 pacientes com
o diagnóstico confirmado por angiografia (`data/pacientes_cardiacos_fase1.csv`). Ele
não tem texto, mas tem a variável `tipo_dor_peito`, e "peito" é justamente o termo em
que o nosso classificador mais se apoia (seção 5). Cruzar as duas coisas mostra quem a
triagem por texto deixa de ver.

In [13]:
pacientes = carregar_pacientes()
print(f"{len(pacientes)} pacientes · {pacientes.doente.mean():.1%} com doença coronariana\n")
display(prevalencia_por(pacientes, "dor_no_peito"))
display(prevalencia_por(pacientes, "sexo_desc"))
display(doentes_assintomaticos(pacientes))

303 pacientes · 45.9% com doença coronariana



,pacientes,doentes,% doentes
dor_no_peito,,,
angina típica,23,7,30.4
angina atípica,50,9,18.0
dor não anginosa,86,18,20.9
assintomático,144,105,72.9


,pacientes,doentes,% doentes
sexo_desc,,,
feminino,97,25,25.8
masculino,206,114,55.3


,doentes,sem_dor_no_peito,% sem dor no peito
sexo_desc,,,
feminino,25,22,88.0
masculino,114,83,72.8
total,139,105,75.5


**Três de cada quatro doentes (75,5%) estão no grupo "assintomático"**: não relataram
dor torácica nenhuma. É também o grupo com a maior prevalência de doença (72,9%), muito
acima de quem relatou angina típica (30,4%). Entre as **mulheres doentes, 22 de 25 (88%)**
não tinham dor no peito.

Isso inverte a premissa do nosso classificador. Ele aprendeu que "peito" puxa para alto
risco; no dado real da Fase 1, **o paciente doente mais comum é o que não fala em peito**.
O viés que medimos no desafio (apresentação atípica classificada como baixo risco) não é
um caso de borda das nossas 70 frases: é o paciente típico do dataset.

O dataset da Fase 1 tem os seus próprios vieses, que a equipe já registrou naquela fase e
que limitam esta leitura:

- **Seleção:** são pacientes encaminhados para cateterismo, não a população geral. Quem
  chegou ali sem dor no peito chegou por outro motivo clínico (ECG, teste de esforço), e a
  prevalência de 72,9% vale para esse filtro, não para a porta de um pronto-socorro.
- **Sexo:** 206 homens e 97 mulheres; com só 25 mulheres doentes, os 88% vêm de um grupo
  pequeno e devem ser lidos como direção, não como taxa.
- **Tempo e lugar:** coleta de 1988, num único centro nos EUA, longe da população do SUS.

**Quem escreve os dados decide o que o modelo enxerga** — nas nossas 70 frases e nos 303
pacientes da Fase 1. Nos dois casos o problema é de composição da base, e não se resolve
trocando o algoritmo.

**Limite de uso.** Este classificador é um exercício acadêmico sobre 70 frases fictícias.
Ele não foi validado clinicamente, não foi testado com pacientes reais e erra casos
graves de forma sistemática (seção 6). Não é dispositivo médico e não deve orientar
decisão sobre pessoa nenhuma.

---
## 9. O que esta parte entrega

- `data/frases_risco.csv` — 70 frases rotuladas (`frase,situacao`), 35 por classe
- `data/frases_desafio.csv` + `data/frases_ampliacao.csv` — conjuntos de estresse e de correção
- `src/classificador.py` — pipeline TF-IDF + LogisticRegression, avaliação, limiar e previsão
- `src/vieses_fase1.py` + `data/pacientes_cardiacos_fase1.csv` — viés medido no dataset da Fase 1
- **Resultados medidos:** 76% no teste sorteado · 83% ±11 na validação cruzada ·
  **38% no conjunto-desafio** · linha de base burra em 50% · recall de alto risco de
  0,83 → 0,94 com o limiar em 0,45 · 75,5% dos doentes da Fase 1 sem dor no peito

O número que resume a Fase 2 não é a acurácia: é a distância entre ela e o
conjunto-desafio. Foi ali que o sistema mostrou o que realmente aprendeu.